In [1]:
import sys
sys.path.append('../..')

import pandas as pd
import networkx as nx
import itertools
import json
import os

from networks.utils.get_rich_monthly_nodes import get_rich_monthly_nodes
from networks.utils.calculate_rich_month_similarity import calculate_rich_month_similarity
from networks.utils.weight_calculation_pca import calculate_pca_weights
from networks.utils.prune_to_average_degree import prune_to_average_degree

from itertools import combinations
import numpy as np
from dtaidistance import dtw

In [2]:
def create_all_global_networks(specifications):

    json_filename = "network_rich_thresholds.json"
    weights_filename = "network_weights.json"

    for city, start, end in specifications:

        print(f"Starting to work on {city} : {start} - {end}")
        
        monthly_nodes = get_rich_monthly_nodes(city, start, end)
        months = list(monthly_nodes.keys())

        dtw_distances = []

        for m1, m2 in combinations(months, 2):
            d1 = np.array(monthly_nodes[m1]['tg_derivatives'], dtype=np.double)
            d2 = np.array(monthly_nodes[m2]['tg_derivatives'], dtype=np.double)

            # z-normalize the sequences (important)
            if d1.std() > 0 and d2.std() > 0:
                d1 = (d1 - d1.mean()) / d1.std()
                d2 = (d2 - d2.mean()) / d2.std()

            dist = dtw.distance(d1, d2)
            dtw_distances.append(dist)

        dtw_mean = np.mean(dtw_distances)
        dtw_std  = np.std(dtw_distances)

        tg_diffs, tn_diffs, tx_diffs, rr_sum_diffs, qq_diffs, hu_diffs, fg_diffs = [], [], [], [], [], [], []

        for m1, m2 in combinations(months, 2):
            tg_diffs.append(abs(monthly_nodes[m1]['mean_tg'] - monthly_nodes[m2]['mean_tg']))
            tn_diffs.append(abs(monthly_nodes[m1]['mean_tn'] - monthly_nodes[m2]['mean_tn']))
            tx_diffs.append(abs(monthly_nodes[m1]['mean_tx'] - monthly_nodes[m2]['mean_tx']))
            rr_sum_diffs.append(abs(monthly_nodes[m1]['rr_sum'] - monthly_nodes[m2]['rr_sum']))
            qq_diffs.append(abs(monthly_nodes[m1]['mean_qq'] - monthly_nodes[m2]['mean_qq']))
            hu_diffs.append(abs(monthly_nodes[m1]['mean_hu'] - monthly_nodes[m2]['mean_hu']))
            fg_diffs.append(abs(monthly_nodes[m1]['mean_fg'] - monthly_nodes[m2]['mean_fg']))

        tg_mean, tg_std = np.mean(tg_diffs), np.std(tg_diffs)
        tn_mean, tn_std = np.mean(tn_diffs), np.std(tn_diffs)
        tx_mean, tx_std = np.mean(tx_diffs), np.std(tx_diffs)
        rr_sum_mean, rr_sum_std = np.mean(rr_sum_diffs), np.std(rr_sum_diffs)
        qq_mean, qq_std = np.mean(qq_diffs), np.std(qq_diffs)
        hu_mean, hu_std = np.mean(hu_diffs), np.std(hu_diffs)
        fg_mean, fg_std = np.mean(fg_diffs), np.std(fg_diffs)

        stats = {
            'dtw_mean': dtw_mean,
            'dtw_std': dtw_std,
            'tg_mean': tg_mean,
            'tg_std': tg_std,
            'tn_mean': tn_mean,
            'tn_std': tn_std,
            'tx_mean': tx_mean,
            'tx_std': tx_std,
            'rr_sum_mean': rr_sum_mean,
            'rr_sum_std': rr_sum_std,
            'qq_mean': qq_mean,
            'qq_std': qq_std,
            'hu_mean': hu_mean,
            'hu_std': hu_std,
            'fg_mean': fg_mean,
            'fg_std': fg_std
        }

        weights, explained_variance_ratio = calculate_pca_weights(monthly_nodes, months)

        json_key = f"{city}_{start}_{end}"
        
        if os.path.exists(weights_filename):
            with open(weights_filename, 'r') as f:
                try:
                    data_store = json.load(f)
                except json.JSONDecodeError:
                    data_store = {}
        else:
            data_store = {}

        data_store[json_key] = (weights, explained_variance_ratio)
        
        with open(json_filename, 'w') as f:
            json.dump(data_store, f, indent=4)

        print(f"Got weights for {json_key}: {weights}")

        print(f"Statistics and weights calculated")

        # --- Network Construction ---
        G = nx.Graph()
        simple_G = nx.Graph()

        # Add nodes to the graph
        for month, data in monthly_nodes.items():
            G.add_node(month, **data)
            simple_G.add_node(month)

        # --- Calculate similarities and add edges ---
        similarity_threshold = 0.3

        for month1, month2 in itertools.combinations(months, 2):
            month1_data = monthly_nodes[month1]
            month2_data = monthly_nodes[month2]
            
            similarity_score = calculate_rich_month_similarity(month1_data, month2_data, stats, weights)
            
            if similarity_score > similarity_threshold:
                G.add_edge(month1, month2, weight=similarity_score)
                simple_G.add_edge(month1, month2, weight = similarity_score)

        # --- Inspect the graph ---
        print(f"Number of nodes: {G.number_of_nodes()}")
        print(f"Number of edges: {G.number_of_edges()}")

        print(f"First network created")

        g_type = 'global'

        graph_path = f"{city}_{g_type}_{start}_{end}_sim_{int(similarity_threshold*100)}.graphml"

        t_avg_deg = int(end[:4]) - int(start[:4]) + 1
        t_avg_deg = t_avg_deg * 3 - 1

        print(f"Starting pruning to avg deg : {t_avg_deg}")

        pruned_graph = prune_to_average_degree(simple_G, target_avg_degree=t_avg_deg)

        if pruned_graph.number_of_edges() > 0:
            weights_pruned = [d['weight'] for u, v, d in pruned_graph.edges(data=True)]
            final_threshold = float(min(weights_pruned))
        else:
            final_threshold = 0.0

        json_key = f"{city}_{start}_{end}"
        
        if os.path.exists(json_filename):
            with open(json_filename, 'r') as f:
                try:
                    data_store = json.load(f)
                except json.JSONDecodeError:
                    data_store = {}
        else:
            data_store = {}

        data_store[json_key] = final_threshold
        
        with open(json_filename, 'w') as f:
            json.dump(data_store, f, indent=4)
            
        print(f"Threshold {final_threshold:.4f} saved for {json_key}")

        G = pruned_graph

        for node, data in G.nodes(data=True):
            try:
                year, month = node.split("-")
                data["year"] = int(year)
                data["month"] = int(month)
            except ValueError:
                data["year"] = None
                data["month"] = None

        pruned_YM_path = f"{city}_{g_type}_pruned_{start}_{end}.graphml"

        nx.write_graphml(G, pruned_YM_path)

        print(f"Network pruned and saved")

In [3]:
specifications = [
    ('Cluj', '1961-01', '1990-12'),
    ('Cluj', '1991-01', '2024-12'),
    ('Cluj', '1961-01', '2024-12'),
    ('Bacskatopolya', '1961-01', '1990-12'),
    ('Bacskatopolya', '1991-01', '2024-12'),
    ('Bacskatopolya', '1961-01', '2024-12'),
    ('Brasov', '1961-01', '1990-12'),
    ('Brasov', '1991-01', '2024-12'),
    ('Brasov', '1961-01', '2024-12'),
    ('Deva', '1961-01', '1990-12'),
    ('Deva', '1991-01', '2024-12'),
    ('Deva', '1961-01', '2024-12'),
    ('Gheorgheni', '1961-01', '1990-12'),
    ('Gheorgheni', '1991-01', '2024-12'),
    ('Gheorgheni', '1961-01', '2024-12'),
    ('Gyor', '1961-01', '1990-12'),
    ('Gyor', '1991-01', '2024-12'),
    ('Gyor', '1961-01', '2024-12'),
    ('Kassa', '1961-01', '1990-12'),
    ('Kassa', '1991-01', '2024-12'),
    ('Kassa', '1961-01', '2024-12'),
    ('Kecskemet', '1961-01', '1990-12'),
    ('Kecskemet', '1991-01', '2024-12'),
    ('Kecskemet', '1961-01', '2024-12'),
    ('Keszthely', '1961-01', '1990-12'),
    ('Keszthely', '1991-01', '2024-12'),
    ('Keszthely', '1961-01', '2024-12'),
    ('Oradea', '1961-01', '1990-12'),
    ('Oradea', '1991-01', '2024-12'),
    ('Oradea', '1961-01', '2024-12'),
    ('Pecs', '1961-01', '1990-12'),
    ('Pecs', '1991-01', '2024-12'),
    ('Pecs', '1961-01', '2024-12')
]
create_all_global_networks(specifications)

Starting to work on Cluj : 1961-01 - 1990-12


ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values